## Configuration


In [ ]:
dbutils.widgets.text("catalog", "training_lab")
dbutils.widgets.text("schema", "student_01")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
ns = f"{catalog}.{schema}"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print("Working namespace:", ns)
from pyspark.sql import functions as F


## Create a small performance fixture


In [ ]:
spark.sql("CREATE OR REPLACE TABLE day18_perf USING DELTA AS\nSELECT id AS event_id, CAST(id % 100 AS INT) AS store_id,\n date_add(DATE'2026-01-01', CAST(id % 365 AS INT)) AS event_date,\n CAST(id % 500 AS DOUBLE) AS amount\nFROM range(100000)")

## Read the plan


In [ ]:
spark.sql("EXPLAIN FORMATTED SELECT store_id, SUM(amount) AS revenue\nFROM day18_perf\nWHERE event_date >= DATE'2026-01-01'\n  AND event_date < DATE'2026-02-01'\nGROUP BY store_id").show(truncate=False)

## Run and inspect


In [ ]:
display(spark.sql("SELECT store_id, SUM(amount) AS revenue\nFROM day18_perf\nWHERE event_date >= DATE'2026-01-01'\n  AND event_date < DATE'2026-02-01'\nGROUP BY store_id"))